# Portfolio Risk Engine — analyse reproductible

## 1. Imports et paramètres

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from risk_engine import (
    ASSET_CLASSES, DEFAULT_TICKERS, HISTORICAL_EVENTS, MARKET_BETAS,
    asset_metrics, compute_risk_table, correlation_stress, download_yahoo_prices,
    historical_event_returns, load_prices, portfolio_metrics, portfolio_returns,
    reverse_stress, risk_contributions, rolling_historical_backtest, simple_returns,
    volatility_stress,
)

BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "portfolio_prices.csv"
META_PATH = BASE_DIR / "data_source.json"
SEED = 42
USE_YAHOO_DOWNLOAD = True  # Passer à True pour télécharger Yahoo Finance dans un environnement connecté.
START_DATE = "2006-01-01"  # Assez tôt pour tester l'épisode de 2008 lorsque l'historique existe.
PORTFOLIO_VALUE = 1_000_000.0
RISK_FREE_RATE = 0.03
N_SIMULATIONS = 20_000
BACKTEST_LOOKBACK = 500

TICKERS = ["AAPL", "MSFT", "NVDA", "TLT", "IEF", "GLD"]
WEIGHTS = np.array([1 / len(TICKERS)] * (len(TICKERS) - 1) + [1 - (len(TICKERS) - 1) / len(TICKERS)])
assert len(TICKERS) >= 6
assert len({ASSET_CLASSES[t] for t in TICKERS}) >= 3
assert np.isclose(WEIGHTS.sum(), 1.0)

## 2. Chargement des cours et provenance

In [ ]:
if USE_YAHOO_DOWNLOAD:
    prices_all = download_yahoo_prices(list(ASSET_CLASSES.keys()), start=START_DATE)
    prices_all.to_csv(CSV_PATH, index_label="Date", date_format="%Y-%m-%d")
    metadata = {
        "kind": "yahoo_finance",
        "source": "Yahoo Finance via yfinance",
        "price_field": "Adjusted Close (Close avec auto_adjust=True)",
        "start_date": str(prices_all.index.min().date()),
        "end_date": str(prices_all.index.max().date()),
        "tickers": list(prices_all.columns),
        "note": "Calendrier aligné et valeurs manquantes intermédiaires propagées vers l'avant.",
    }
    META_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
else:
    prices_all = load_prices(CSV_PATH)
    metadata = json.loads(META_PATH.read_text(encoding="utf-8")) if META_PATH.exists() else {"source": "Inconnue"}

available = [ticker for ticker in TICKERS if ticker in prices_all.columns]
missing = sorted(set(TICKERS) - set(available))
if missing:
    raise ValueError(f"Tickers absents du CSV : {missing}")
prices = prices_all[available].ffill().dropna(how="any")
returns = simple_returns(prices)
weights = WEIGHTS.copy()

print("Source :", metadata.get("source", "non renseignée"))
print("Champ de prix :", metadata.get("price_field", "à vérifier"))
print("Période :", prices.index.min().date(), "→", prices.index.max().date())
print("Cours :", prices.shape, "| rendements communs :", returns.shape)
print("Tickers :", available)
if metadata.get("kind") != "yahoo_finance":
    print("ATTENTION : jeu de démonstration synthétique. Remplacer par les cours Yahoo Finance avant le rendu.")

Source : Yahoo Finance via yfinance
Champ de prix : Adjusted Close (Close avec auto_adjust=True)
Période : 2007-04-10 → 2026-10-20
Cours : (4915, 6) | rendements communs : (4914, 6)
Tickers : ['AAPL', 'MSFT', 'NVDA', 'TLT', 'IEF', 'GLD']


## 3. Prix, rendements et statistiques de risque

In [ ]:
normalized_prices = prices / prices.iloc[0] * 100
fig = px.line(normalized_prices, labels={"value": "Base 100", "variable": "Ticker", "Date": "Date"}, title="Évolution normalisée des prix")
fig.update_layout(height=450)
fig.show()

asset_stats = asset_metrics(returns, risk_free_rate=RISK_FREE_RATE)
p_ret = portfolio_returns(returns, weights)
portfolio_stats = portfolio_metrics(p_ret, risk_free_rate=RISK_FREE_RATE)
print("Statistiques du portefeuille équipondéré :")
for metric, value in portfolio_stats.items():
    print(f"{metric}: {value:.2%}" if metric != "Sharpe" else f"{metric}: {value:.3f}")
asset_stats.style.format({
    "Rendement annualisé": "{:.2%}",
    "Volatilité annualisée": "{:.2%}",
    "Sharpe": "{:.2f}",
    "Drawdown maximal": "{:.2%}",
})

Statistiques du portefeuille équipondéré :
Rendement annualisé: 17.41%
Volatilité annualisée: 15.12%
Sharpe: 0.954
Drawdown maximal: -36.95%


,Classe d'actifs,Rendement annualisé,Volatilité annualisée,Sharpe,Drawdown maximal
Ticker,,,,,
AAPL,Actions,24.57%,31.16%,0.69,-60.87%
MSFT,Actions,16.82%,28.23%,0.49,-57.94%
NVDA,Actions,31.58%,48.89%,0.58,-85.08%
TLT,Obligations US,2.68%,15.09%,-0.02,-48.35%
IEF,Obligations US,3.01%,6.94%,0.00,-23.92%
GLD,Or,8.86%,18.19%,0.32,-45.56%


## 4. Corrélations et diversification

In [ ]:
corr_all = returns.corr()
fig = px.imshow(corr_all, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="RdBu_r", aspect="auto", title="Corrélations sur toute la période")
fig.update_layout(height=500)
fig.show()

crisis_returns = returns.loc["2020-02-19":"2020-04-30"]
if len(crisis_returns) >= 10:
    corr_crisis = crisis_returns.corr()
    fig = px.imshow(corr_crisis, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="RdBu_r", aspect="auto", title="Corrélations pendant le choc Covid-19")
    fig.update_layout(height=500)
    fig.show()
    tri = np.triu_indices(len(TICKERS), 1)
    mean_corr_all = float(corr_all.to_numpy()[tri].mean())
    mean_corr_crisis = float(corr_crisis.to_numpy()[tri].mean())
    print(f"Corrélation moyenne hors diagonale — période entière : {mean_corr_all:.3f}")
    print(f"Corrélation moyenne hors diagonale — Covid-19 : {mean_corr_crisis:.3f}")
else:
    corr_crisis = None
    mean_corr_all = float("nan")
    mean_corr_crisis = float("nan")
    print("L'épisode Covid-19 n'est pas disponible dans la période du CSV.")

contributions = risk_contributions(returns, weights)
contributions

Corrélation moyenne hors diagonale — période entière : 0.122
Corrélation moyenne hors diagonale — Covid-19 : 0.095


,Ticker,Poids (%),Contribution au risque (%),Contribution vol. quotidienne
0,NVDA,16.666667,44.994009,0.004285
1,AAPL,16.666667,24.950622,0.002376
2,MSFT,16.666667,23.085818,0.002198
3,GLD,16.666667,5.746162,0.000547
4,TLT,16.666667,0.835962,0.000080
5,IEF,16.666667,0.387429,0.000037


## 5. VaR et Expected Shortfall : trois méthodes, deux horizons

In [ ]:
risk_table = compute_risk_table(
    asset_returns=returns,
    weights=weights,
    portfolio_value=PORTFOLIO_VALUE,
    horizons=(1, 10),
    confidence_levels=(0.95, 0.99),
    n_simulations=N_SIMULATIONS,
    seed=SEED,
)
print(f"Valeur du portefeuille : {PORTFOLIO_VALUE:,.0f} €")
print(f"Période : {returns.index.min().date()} — {returns.index.max().date()}")
risk_table.style.format({col: "{:,.0f} €" for col in risk_table.columns if "(€)" in col})

Valeur du portefeuille : 1,000,000 €
Période : 2007-04-11 — 2026-10-20


,Méthode,Horizon (jours),VaR 95 % (€),ES 95 % (€),VaR 99 % (€),ES 99 % (€)
0,Historique,1,"14,339 €","21,500 €","25,174 €","33,147 €"
1,Paramétrique (normal),1,"14,927 €","18,906 €","21,416 €","24,643 €"
2,Monte Carlo,1,"14,911 €","18,544 €","20,809 €","23,594 €"
3,Historique,10,"41,223 €","59,495 €","70,997 €","88,894 €"
4,Paramétrique (normal),10,"42,166 €","54,749 €","62,688 €","72,892 €"
5,Monte Carlo,10,"41,917 €","53,473 €","60,458 €","69,304 €"


## 6. Stress tests

In [ ]:
baseline_var99 = float(risk_table.query("Méthode == 'Historique' and `Horizon (jours)` == 1")["VaR 99 % (€)"].iloc[0])
baseline_es99 = float(risk_table.query("Méthode == 'Historique' and `Horizon (jours)` == 1")["ES 99 % (€)"].iloc[0])

def ratio_to_baseline(loss, baseline):
    return loss / baseline if np.isfinite(baseline) and abs(baseline) > 1e-10 else np.nan

def append_stress(name, period, portfolio_return_pct, loss_eur, worst_asset):
    stress_rows.append({
        "Scénario": name,
        "Période / type": period,
        "Rendement portefeuille (%)": portfolio_return_pct,
        "Perte stressée (€)": loss_eur,
        "Perte / VaR historique 99 %": ratio_to_baseline(loss_eur, baseline_var99),
        "Perte / ES historique 99 %": ratio_to_baseline(loss_eur, baseline_es99),
        "Actif ou facteur le plus pénalisant": worst_asset,
    })

stress_rows = []
stress_asset_impacts = {}
for event_name in HISTORICAL_EVENTS:
    event_ret, actual_start, actual_end = historical_event_returns(prices, TICKERS, event_name)
    if event_ret is None:
        append_stress(event_name, "non disponible", np.nan, np.nan, "—")
        continue
    weighted_contribution = weights * event_ret.to_numpy()
    p_return = float(weighted_contribution.sum())
    worst = TICKERS[int(np.argmin(weighted_contribution))]
    append_stress(event_name, f"{actual_start} — {actual_end}", p_return * 100, -p_return * PORTFOLIO_VALUE, worst)
    stress_asset_impacts[event_name] = pd.DataFrame({"Ticker": TICKERS, "Rendement épisode (%)": event_ret.to_numpy() * 100, "Contribution portefeuille (%)": weighted_contribution * 100, "Contribution (€)": weighted_contribution * PORTFOLIO_VALUE})

# Scénario hypothétique : baisse personnalisée par actif (modifiable).
custom_shocks_pct = np.array([-15, -15, -25, -5, -5, 0], dtype=float)
custom_ret = custom_shocks_pct / 100
custom_contrib = weights * custom_ret
append_stress("Choc hypothétique", "chocs saisis", custom_contrib.sum() * 100, -custom_contrib.sum() * PORTFOLIO_VALUE, TICKERS[int(np.argmin(custom_contrib))])

corr_result = correlation_stress(returns, weights, PORTFOLIO_VALUE, target_correlation=0.90, confidence=0.99, horizon=1)
vol_result = volatility_stress(returns, weights, PORTFOLIO_VALUE, multiplier=2.0, confidence=0.99, horizon=1)
append_stress("Corrélation stressée ρ=0,90", "VaR paramétrique 99 % stressée", np.nan, corr_result["VaR (€)"], "Corrélations")
append_stress("Volatilités × 2", "VaR paramétrique 99 % stressée", np.nan, vol_result["VaR (€)"], "Volatilité globale")
reverse = reverse_stress(TICKERS, weights, target_loss_fraction=0.20)
append_stress("Reverse stress −20 %", f"Choc marché {reverse['Choc marché (%)']:.2f} %", -20.0, 0.20 * PORTFOLIO_VALUE, TICKERS[int(np.argmin(weights * reverse["Chocs actifs (%)"].to_numpy() / 100))])

stress_table = pd.DataFrame(stress_rows)
stress_table

,Scénario,Période / type,Rendement portefeuille (%),Perte stressée (€),Perte / VaR historique 99 %,Perte / ES historique 99 %,Actif ou facteur le plus pénalisant
0,Crise financière 2008 (sept. 2008–mars 2009),2008-09-02 — 2009-03-31,-8.307427,83074.265997,3.299978,2.506220,AAPL
1,Covid-19 (19 fév.–23 mars 2020),2020-02-19 — 2020-03-23,-12.240342,122403.417805,4.862259,3.692719,NVDA
2,Choc inflation/taux (janv.–oct. 2022),2022-01-03 — 2022-10-12,-28.274096,282740.962999,11.231384,8.529850,NVDA
3,Choc hypothétique,chocs saisis,-10.833333,108333.333333,4.303350,3.268246,NVDA
4,"Corrélation stressée ρ=0,90",VaR paramétrique 99 % stressée,NaN,34082.805982,1.353879,1.028225,Corrélations
5,Volatilités × 2,VaR paramétrique 99 % stressée,NaN,43568.892162,1.730697,1.314405,Volatilité globale
6,Reverse stress −20 %,Choc marché -30.61 %,-20.000000,200000.000000,7.944645,6.033686,NVDA


In [7]:
print("Détail du scénario hypothétique :")
pd.DataFrame({
    "Ticker": TICKERS,
    "Poids (%)": weights * 100,
    "Choc saisi (%)": custom_shocks_pct,
    "Contribution au rendement (%)": custom_contrib * 100,
    "Contribution (€)": custom_contrib * PORTFOLIO_VALUE,
})

print("Stress de corrélation :", corr_result)
print("Stress de volatilité :", vol_result)
print("Reverse stress : bêta effectif =", round(float(reverse["Bêta effectif"]), 3), "; choc marché requis (%) =", round(float(reverse["Choc marché (%)"]), 2))
print("Chocs inverses par actif :")
reverse["Chocs actifs (%)"]


Détail du scénario hypothétique :
Stress de corrélation : {'Volatilité quotidienne': 0.014967382869892324, 'Volatilité annualisée': np.float64(0.2375998371073399), 'VaR (€)': 34082.805981879785, 'ES (€)': 39154.74832874773}
Stress de volatilité : {'Volatilité quotidienne': 0.019045055984154253, 'Volatilité annualisée': np.float64(0.3023308910362479), 'VaR (€)': 43568.89216227653, 'ES (€)': 50022.62069880039}
Reverse stress : bêta effectif = 0.653 ; choc marché requis (%) = -30.61
Chocs inverses par actif :


AAPL   -36.734694
MSFT   -33.673469
NVDA   -52.040816
TLT      3.061224
IEF      2.448980
GLD     -3.061224
Name: Choc (%), dtype: float64

## 7. Backtesting rolling de la VaR historique à 99 %

In [8]:
backtest, backtest_stats = rolling_historical_backtest(
    p_ret,
    portfolio_value=PORTFOLIO_VALUE,
    lookback=BACKTEST_LOOKBACK,
    confidence=0.99,
    test_days=250,
)
print(pd.Series(backtest_stats))
fig = go.Figure()
fig.add_trace(go.Scatter(x=backtest.index, y=backtest["VaR historique (€)"], mode="lines", name="VaR 99 %"))
fig.add_trace(go.Scatter(x=backtest.index, y=backtest["Perte réalisée (€)"], mode="lines", name="Perte réalisée"))
exceptions = backtest[backtest["Exception"]]
fig.add_trace(go.Scatter(x=exceptions.index, y=exceptions["Perte réalisée (€)"], mode="markers", name="Exceptions", marker={"size": 9, "symbol": "x"}))
fig.update_layout(title="Backtest hors échantillon — dernier bloc de 250 jours", xaxis_title="Date", yaxis_title="Perte (€)", height=450)
fig.show()
backtest.tail(10)


Exceptions                                          1
Observations                                      250
Fréquence observée                              0.004
Fréquence attendue                               0.01
LR Kupiec                                    1.176491
p-value                                      0.278071
Couverture non rejetée à 5 %                     True
Feu tricolore                   Vert (0–4 exceptions)
Fenêtre d'estimation (jours)                      500
Date de début du test                      2025-10-22
Date de fin du test                        2026-10-20
dtype: object


,VaR historique (€),Perte réalisée (€),Exception
Date,,,
2026-10-07,23167.45043,2673.303629,False
2026-10-08,23167.45043,1892.675573,False
2026-10-09,23167.45043,-4021.087828,False
2026-10-12,23167.45043,-1038.351681,False
2026-10-13,23167.45043,-3154.113085,False
2026-10-14,23167.45043,8491.695331,False
2026-10-15,23167.45043,-4713.405903,False
2026-10-16,23167.45043,4886.026973,False
2026-10-19,23167.45043,-592.456954,False


## 8. Réponses synthétiques aux questions du projet

In [9]:
# 1. Quel actif est le plus volatil ?
most_volatile = asset_stats["Volatilité annualisée"].idxmax()
print(f"1. Actif le plus volatil : {most_volatile} ({asset_stats.loc[most_volatile, 'Volatilité annualisée']:.2%} annualisé).")

# 2. La VaR passe-t-elle le backtest ?
print(f"2. Exceptions : {backtest_stats['Exceptions']} / {backtest_stats['Observations']}; feu : {backtest_stats['Feu tricolore']}; p-value Kupiec : {backtest_stats['p-value']:.4f}.")
print("   Lecture : couverture non rejetée à 5 % =", backtest_stats["Couverture non rejetée à 5 %"], ". Le feu tricolore est spécifique au test à 99 % sur 250 jours.")

# 3. Les corrélations changent-elles en période de crise ?
print(f"3. Corrélation moyenne hors diagonale : période entière {mean_corr_all:.3f}; Covid-19 {mean_corr_crisis:.3f}. Interpréter l'écart avec prudence : sous-période plus courte.")

# 4. Pourquoi la VaR historique et paramétrique diffèrent-elles ?
hist99 = float(risk_table.query("Méthode == 'Historique' and `Horizon (jours)` == 1")["VaR 99 % (€)"].iloc[0])
param99 = float(risk_table.query("Méthode == 'Paramétrique (normal)' and `Horizon (jours)` == 1")["VaR 99 % (€)"].iloc[0])
print(f"4. VaR 99 % historique = {hist99:,.0f} € ; paramétrique = {param99:,.0f} €. L'historique utilise un quantile empirique, tandis que la méthode paramétrique suppose une loi normale ; fenêtre, asymétrie et queues épaisses peuvent expliquer l'écart.")

# 5. Pourquoi la VaR Monte Carlo diffère-t-elle de la VaR historique ?
mc99 = float(risk_table.query("Méthode == 'Monte Carlo' and `Horizon (jours)` == 1")["VaR 99 % (€)"].iloc[0])
print(f"5. VaR 99 % Monte Carlo = {mc99:,.0f} € (historique = {hist99:,.0f} €). Monte Carlo échantillonne le modèle multivarié ajusté ; le résultat dépend des paramètres estimés, de la loi supposée et de la seed.")

# 6. Pourquoi l'Expected Shortfall complète-t-elle la VaR ?
es99 = float(risk_table.query("Méthode == 'Historique' and `Horizon (jours)` == 1")["ES 99 % (€)"].iloc[0])
print(f"6. ES 99 % historique = {es99:,.0f} €. La VaR localise le seuil de queue ; l'ES moyenne les pertes au-delà du seuil et renseigne sur leur gravité.")

# 7. Que se passe-t-il si le portefeuille est plus concentré ?
hhi = float(np.sum(weights ** 2))
effective_positions = 1.0 / hhi
weighted_average_vol = float(np.sum(weights * asset_stats["Volatilité annualisée"].reindex(TICKERS).to_numpy()))
print(f"7. HHI = {hhi:.3f}; nombre effectif de positions = {effective_positions:.2f}; volatilité du portefeuille = {portfolio_stats['Volatilité annualisée']:.2%}; moyenne pondérée des volatilités individuelles = {weighted_average_vol:.2%}. Une concentration accrue réduit le bénéfice potentiel de diversification, selon les corrélations.")

# 8. Quel stress est le pire et quel actif le pilote ?
valid_stress = stress_table.dropna(subset=["Perte stressée (€)"])
worst = valid_stress.loc[valid_stress["Perte stressée (€)"].idxmax()]
print(f"8. Perte la plus élevée parmi les scénarios chiffrés : {worst['Scénario']} ({worst['Perte stressée (€)']:,.0f} €). Actif / facteur indiqué par le tableau : {worst['Actif ou facteur le plus pénalisant']}.")
print("   Pour les scénarios historiques et hypothétiques, consulter le détail par actif pour identifier la contribution la plus défavorable.")


1. Actif le plus volatil : NVDA (48.89% annualisé).
2. Exceptions : 1 / 250; feu : Vert (0–4 exceptions); p-value Kupiec : 0.2781.
   Lecture : couverture non rejetée à 5 % = True . Le feu tricolore est spécifique au test à 99 % sur 250 jours.
3. Corrélation moyenne hors diagonale : période entière 0.122; Covid-19 0.095. Interpréter l'écart avec prudence : sous-période plus courte.
4. VaR 99 % historique = 25,174 € ; paramétrique = 21,416 €. L'historique utilise un quantile empirique, tandis que la méthode paramétrique suppose une loi normale ; fenêtre, asymétrie et queues épaisses peuvent expliquer l'écart.
5. VaR 99 % Monte Carlo = 20,809 € (historique = 25,174 €). Monte Carlo échantillonne le modèle multivarié ajusté ; le résultat dépend des paramètres estimés, de la loi supposée et de la seed.
6. ES 99 % historique = 33,147 €. La VaR localise le seuil de queue ; l'ES moyenne les pertes au-delà du seuil et renseigne sur leur gravité.
7. HHI = 0.167; nombre effectif de positions = 6.